In [ ]:
!apt-get update && apt-get install -y build-essential
!pip install wisardpkg==1.6.3

In [ ]:
!pip install wisardpkg==1.6.3 --no-build-isolation


In [ ]:
!pip install codecarbon

In [ ]:
!pip install -U ultralytics

In [ ]:
# ============================================================
# DWN vs WiSARD vs YOLO26n
# SAR OIL SPILL CLASSIFICATION
# ============================================================

import os
import random
import shutil
import zipfile
import time
import glob
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from ultralytics import YOLO

# ============================================================
# CONFIGURAÇÕES
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Tamanho pequeno para treinamento rápido
IMG_SIZE = 128

# Batch
BATCH_SIZE = 128

# Épocas
DWN_EPOCHS = 15
WISARD_EPOCHS = 55
YOLO_EPOCHS = 15

# Classes
CLASS_NAMES = [
    "NO OIL",
    "OIL"
]

In [ ]:
# ============================================================
# DOWNLOAD DO DATASET DIRETAMENTE DO KAGGLE
# ============================================================

import os
import glob
import zipfile
import requests

DATASET_ROOT = "./dataset"

os.makedirs(DATASET_ROOT, exist_ok=True)

# Dataset do Kaggle
DATASET_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "harikrishnacs/sentinel-1-sar-oil-spill-detection-dataset"
)

ZIP_PATH = os.path.join(
    DATASET_ROOT,
    "sentinel-1-sar-oil-spill-detection-dataset.zip"
)

# Procurar ZIP já existente
zip_files = glob.glob(
    DATASET_ROOT + "/**/*.zip",
    recursive=True
)

print("ZIPs encontrados:")

for f in zip_files:
    print(" -", f)


# ============================================================
# BAIXAR SE NÃO EXISTIR
# ============================================================

if len(zip_files) > 0:

    ZIP_PATH = zip_files[0]

    print("\nUsando ZIP existente:")
    print(ZIP_PATH)

else:

    print("\nNenhum ZIP encontrado.")
    print("Baixando diretamente do Kaggle...")

    response = requests.get(
        DATASET_URL,
        stream=True,
        timeout=300
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Erro ao baixar dataset do Kaggle. "
            f"HTTP {response.status_code}"
        )

    total_size = int(
        response.headers.get("content-length", 0)
    )

    downloaded = 0

    with open(ZIP_PATH, "wb") as f:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if chunk:

                f.write(chunk)
                downloaded += len(chunk)

                if total_size > 0:

                    percent = (
                        downloaded / total_size
                    ) * 100

                    print(
                        f"\rDownload: {percent:.1f}%",
                        end=""
                    )

    print("\n\nDownload concluído.")
    print("Arquivo:", ZIP_PATH)


# ============================================================
# EXTRAIR DATASET
# ============================================================

EXTRACT_DIR = os.path.join(
    DATASET_ROOT,
    "extracted"
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)

print("\nExtraindo dataset...")

with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as zip_ref:

    zip_ref.extractall(EXTRACT_DIR)

print("Extração concluída.")

# ============================================================
# MOSTRAR ESTRUTURA
# ============================================================

print("\nArquivos encontrados:")

for root, dirs, files in os.walk(EXTRACT_DIR):

    level = root.replace(
        EXTRACT_DIR,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    for file in files[:10]:

        print(
            f"{indent}    {file}"
        )

    if len(files) > 10:
        print(
            f"{indent}    ... "
            f"({len(files)} arquivos)"
        )

print("\nDataset pronto em:")
print(EXTRACT_DIR)

In [ ]:
# ============================================================
# EXTRAIR DATASET
# ============================================================

EXTRACT_DIR = "./dataset/extracted"

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)


# Verifica se já existe conteúdo
if len(os.listdir(EXTRACT_DIR)) == 0:

    print("Extraindo...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            EXTRACT_DIR
        )

    print("Extração concluída.")

else:

    print(
        "Dataset já estava extraído."
    )

In [ ]:
# ============================================================
# ENCONTRAR CLASS_0 E CLASS_1
# ============================================================

class0_dirs = glob.glob(
    EXTRACT_DIR + "/**/Class_0",
    recursive=True
)

class1_dirs = glob.glob(
    EXTRACT_DIR + "/**/Class_1",
    recursive=True
)


if len(class0_dirs) == 0:
    raise RuntimeError(
        "Não encontrei a pasta Class_0"
    )

if len(class1_dirs) == 0:
    raise RuntimeError(
        "Não encontrei a pasta Class_1"
    )


CLASS0_DIR = class0_dirs[0]
CLASS1_DIR = class1_dirs[0]


print("Class_0:", CLASS0_DIR)
print("Class_1:", CLASS1_DIR)

In [ ]:
import os
import random

# ============================================================
# CARREGAR IMAGENS
# ============================================================

EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
)

def get_images(directory):
    files = []
    for root, dirs, filenames in os.walk(directory):
        for filename in filenames:
            if filename.lower().endswith(EXTENSIONS):
                files.append(os.path.join(root, filename))
    return files

no_oil_images = get_images(CLASS0_DIR)
oil_images = get_images(CLASS1_DIR)

print("--- Antes do Balanceamento ---")
print("NO OIL:", len(no_oil_images))
print("OIL:", len(oil_images))

if len(no_oil_images) == 0 or len(oil_images) == 0:
    raise RuntimeError("Uma das classes está vazia.")

# ============================================================
# BALANCEAMENTO DAS CLASSES (Undersampling)
# ============================================================

# Encontra o tamanho da menor classe
min_len = min(len(no_oil_images), len(oil_images))

# Seleciona imagens aleatórias de ambas as listas até o limite da menor classe
no_oil_images = random.sample(no_oil_images, min_len)
oil_images = random.sample(oil_images, min_len)

print("\n--- Depois do Balanceamento ---")
print("NO OIL:", len(no_oil_images))
print("OIL:", len(oil_images))

In [ ]:
# ============================================================
# SPLIT ÚNICO
# ============================================================

all_paths = (
    no_oil_images +
    oil_images
)

all_labels = (
    [0] * len(no_oil_images) +
    [1] * len(oil_images)
)


all_paths = np.array(
    all_paths
)

all_labels = np.array(
    all_labels
)


# ------------------------------------------------------------
# 70% treino
# 15% validação
# 15% teste
# ------------------------------------------------------------

train_paths, temp_paths, \
train_labels, temp_labels = train_test_split(

    all_paths,
    all_labels,

    test_size=0.30,

    stratify=all_labels,

    random_state=SEED
)


val_paths, test_paths, \
val_labels, test_labels = train_test_split(

    temp_paths,
    temp_labels,

    test_size=0.50,

    stratify=temp_labels,

    random_state=SEED
)


print(
    "TRAIN:",
    len(train_paths)
)

print(
    "VAL:",
    len(val_paths)
)

print(
    "TEST:",
    len(test_paths)
)


print()

print(
    "Train - NO OIL:",
    np.sum(train_labels == 0)
)

print(
    "Train - OIL:",
    np.sum(train_labels == 1)
)

print()

print(
    "Test - NO OIL:",
    np.sum(test_labels == 0)
)

print(
    "Test - OIL:",
    np.sum(test_labels == 1)
)

In [ ]:
# ============================================================
# CURA DO VÍCIO DA DWN: OVERSAMPLING DE ÓLEO
# ============================================================
from imblearn.over_sampling import RandomOverSampler

print("\n--- APLICANDO OVERSAMPLING ---")

# O RandomOverSampler exige uma matriz 2D.
# Como nossos caminhos (paths) são uma lista 1D, fazemos um reshape rápido:
train_paths_2d = train_paths.reshape(-1, 1)

# Cria o algoritmo de balanceamento
ros = RandomOverSampler(random_state=SEED)

# Clona os caminhos e labels das imagens de Óleo até empatar com as de Água
train_paths_bal_2d, train_labels_bal = ros.fit_resample(train_paths_2d, train_labels)

# Volta os caminhos para o formato original (lista simples/1D)
train_paths_bal = train_paths_bal_2d.flatten()

print("Novo Train (BALANCEADO) - NO OIL:", np.sum(train_labels_bal == 0))
print("Novo Train (BALANCEADO) - OIL:", np.sum(train_labels_bal == 1))
print("------------------------------\n")

In [ ]:
# ============================================================
# CRIAR DATASET PARA YOLO
# ============================================================

YOLO_DATASET = "./sar_yolo"

if os.path.exists(
    YOLO_DATASET
):

    shutil.rmtree(
        YOLO_DATASET
    )


for split in [
    "train",
    "val",
    "test"
]:

    for class_name in [
        "no_oil",
        "oil"
    ]:

        os.makedirs(
            os.path.join(
                YOLO_DATASET,
                split,
                class_name
            ),
            exist_ok=True
        )


def copy_split(
    paths,
    labels,
    split
):

    counters = {
        0: 0,
        1: 0
    }


    for path, label in zip(
        paths,
        labels
    ):

        class_name = (
            "no_oil"
            if label == 0
            else "oil"
        )


        destination_dir = os.path.join(
            YOLO_DATASET,
            split,
            class_name
        )


        extension = os.path.splitext(
            path
        )[1]


        new_name = (
            f"{label}_"
            f"{counters[label]}_"
            f"{os.path.basename(path)}"
        )


        destination = os.path.join(
            destination_dir,
            new_name
        )


        shutil.copy2(
            path,
            destination
        )


        counters[label] += 1


copy_split(
    train_paths,
    train_labels,
    "train"
)

copy_split(
    val_paths,
    val_labels,
    "val"
)

copy_split(
    test_paths,
    test_labels,
    "test"
)


print(
    "Dataset YOLO criado em:",
    YOLO_DATASET
)

In [ ]:
# ============================================================
# DWN SIMPLIFICADA
# ============================================================

class ThermometerEncoder:

    def __init__(
        self,
        bins=4
    ):

        self.bins = bins


    def encode(
        self,
        x
    ):

        # x entre 0 e 1
        thresholds = torch.linspace(
            0.2,
            0.8,
            self.bins,
            device=x.device
        )

        bits = (
            x.unsqueeze(-1)
            > thresholds
        ).float()


        return bits.reshape(
            x.shape[0],
            -1
        )


class LUTLayer(nn.Module):

    def __init__(
        self,
        input_bits,
        num_luts,
        lut_inputs=4
    ):

        super().__init__()

        self.input_bits = input_bits

        self.num_luts = num_luts

        self.lut_inputs = lut_inputs


        # conexões fixas
        self.register_buffer(

            "connections",

            torch.randint(
                0,
                input_bits,
                (
                    num_luts,
                    lut_inputs
                )
            )
        )


        # LUT diferenciável durante treinamento
        self.table = nn.Parameter(

            torch.randn(
                num_luts,
                2 ** lut_inputs
            )
            * 0.1

        )


    def forward(
        self,
        x
    ):

        # ----------------------------------------------------
        # Selecionar bits
        # ----------------------------------------------------

        selected = x[
            :,
            self.connections
        ]


        # ----------------------------------------------------
        # endereço binário
        # ----------------------------------------------------

        powers = (
            2 ** torch.arange(
                self.lut_inputs,
                device=x.device
            )
        )


        address = (
            selected.long()
            * powers
        ).sum(
            dim=2
        )


        # ----------------------------------------------------
        # LUT
        # ----------------------------------------------------

        output = self.table[
            torch.arange(
                self.num_luts,
                device=x.device
            ).unsqueeze(0),
            address
        ]


        return torch.tanh(
            output
        )


class SimpleDWN(nn.Module):

    def __init__(
        self,
        image_size=64,
        thermometer_bins=2,
        lut_inputs=8,
        luts1=256,
        luts2=64,
        classes=2
    ):

        super().__init__()


        self.image_size = image_size


        # grayscale
        input_pixels = (
            image_size *
            image_size
        )


        self.encoder = ThermometerEncoder(
            thermometer_bins
        )


        encoded_bits = (
            input_pixels *
            thermometer_bins
        )


        self.lut1 = LUTLayer(
            encoded_bits,
            luts1,
            lut_inputs
        )


        self.lut2 = LUTLayer(
            luts1,
            luts2,
            lut_inputs
        )


        self.classifier = nn.Linear(
            luts2,
            classes
        )


    def forward(
        self,
        x
    ):

        # grayscale
        if x.shape[1] == 3:

            x = (
                0.299 * x[:, 0:1]
                + 0.587 * x[:, 1:2]
                + 0.114 * x[:, 2:3]
            )


        x = x.flatten(
            start_dim=1
        )


        # normalizar
        x = (
            x - x.min(
                dim=1,
                keepdim=True
            )[0]
        )


        x = x / (
            x.max(
                dim=1,
                keepdim=True
            )[0]
            + 1e-6
        )


        x = self.encoder.encode(
            x
        )


        x = self.lut1(
            x
        )


        # binarização suave
        x = (
            x > 0
        ).float()


        x = self.lut2(
            x
        )


        return self.classifier(
            x
        )

# ============================================================
# ARQUITETURA: Multi-Scale LBP-Wisard (A Carta na Manga)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleLBPWisard(nn.Module):
    def __init__(self, classes=2, num_scales=4):
        super().__init__()
        self.num_scales = num_scales

        # 4 Filtros (Escalas) x 256 endereços x 2 classes = 2.048 parâmetros totais!
        self.rams = nn.Parameter(torch.randn(num_scales, 256, classes) * 0.01)

        # Pesos do LBP clássico
        powers = torch.tensor([
            128, 64, 32,
             1,   0, 16,
             2,   4,  8
        ], dtype=torch.float32).view(1, 9, 1)
        self.register_buffer("powers", powers)

    def forward(self, x):
        if x.shape[1] == 3:
            x = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]

        batch_size = x.size(0)
        final_votes = 0

        # A rede aplica os 4 filtros LBP ao mesmo tempo, varrendo do micro ao macro
        for scale in range(self.num_scales):
            dilation = scale + 1 # Dilatação: 1, 2, 3 e 4

            # O 'dilation' afasta os vizinhos, como uma lente de aumento na imagem
            patches = F.unfold(x, kernel_size=3, padding=dilation, dilation=dilation)

            center_pixel = patches[:, 4:5, :]
            binary_bits = (patches >= center_pixel).float()

            addresses = (binary_bits * self.powers).sum(dim=1).long()

            # Busca os votos apenas no bloco de memória (RAM) equivalente àquela escala
            votes = self.rams[scale][addresses]

            final_votes = final_votes + votes.sum(dim=1)

        return final_votes

In [ ]:
# ============================================================
# DATASET PYTORCH
# ============================================================

class SARDataset(Dataset):

    def __init__(
        self,
        paths,
        labels,
        img_size=64
    ):

        self.paths = list(paths)

        self.labels = list(labels)

        self.img_size = img_size


    def __len__(
        self
    ):

        return len(
            self.paths
        )


    def __getitem__(
        self,
        idx
    ):

        path = self.paths[idx]

        label = self.labels[idx]


        image = Image.open(
            path
        ).convert(
            "L"
        )


        image = image.resize(
            (
                self.img_size,
                self.img_size
            )
        )


        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0


        image = torch.tensor(
            image
        ).unsqueeze(
            0
        )


        return (
            image,
            torch.tensor(
                label,
                dtype=torch.long
            ),
            path
        )


train_dataset = SARDataset(
    train_paths_bal,
    train_labels_bal,
    IMG_SIZE
)

val_dataset = SARDataset(
    val_paths,
    val_labels,
    IMG_SIZE
)

test_dataset = SARDataset(
    test_paths,
    test_labels,
    IMG_SIZE
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)


print(
    "DWN dataset:",
    len(train_dataset),
    len(val_dataset),
    len(test_dataset)
)


In [ ]:
# ============================================================
# TREINAMENTO DWN
# ============================================================

dwn = SimpleDWN(
    image_size=IMG_SIZE,
    thermometer_bins=2,
    lut_inputs=4,
    luts1=256,
    luts2=64,
    classes=2
).to(device)


criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(
    dwn.parameters(),
    lr=0.003
)


print(
    "Parâmetros DWN:",
    sum(
        p.numel()
        for p in dwn.parameters()
    )
)


best_val_acc = 0

best_state = None


for epoch in range(
    DWN_EPOCHS
):

    dwn.train()


    running_loss = 0

    correct = 0

    total = 0


    for images, labels, paths in train_loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        optimizer.zero_grad()


        scores = dwn(
            images
        )


        loss = criterion(
            scores,
            labels
        )


        loss.backward()

        optimizer.step()


        running_loss += (
            loss.item()
            * labels.size(0)
        )


        predictions = torch.argmax(
            scores,
            dim=1
        )


        correct += (
            predictions == labels
        ).sum().item()


        total += labels.size(0)


    train_loss = (
        running_loss / total
    )

    train_acc = (
        correct / total
    )


    # --------------------------------------------------------
    # validação
    # --------------------------------------------------------

    dwn.eval()


    val_correct = 0

    val_total = 0


    with torch.no_grad():

        for images, labels, paths in val_loader:

            images = images.to(
                device
            )

            labels = labels.to(
                device
            )


            scores = dwn(
                images
            )


            predictions = torch.argmax(
                scores,
                dim=1
            )


            val_correct += (
                predictions == labels
            ).sum().item()


            val_total += labels.size(0)


    val_acc = (
        val_correct /
        val_total
    )


    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_state = copy.deepcopy(
            dwn.state_dict()
        )


    print(
        f"Epoch {epoch+1:02d}/{DWN_EPOCHS} "
        f"Loss={train_loss:.4f} "
        f"Train={train_acc*100:.2f}% "
        f"Val={val_acc*100:.2f}%"
    )


# recuperar melhor modelo
if best_state is not None:

    dwn.load_state_dict(
        best_state
    )


print(
    "\nMelhor validação:",
    f"{best_val_acc*100:.2f}%"
)

In [ ]:
# ============================================================
# FUNÇÃO DE AVALIAÇÃO DWN
# ============================================================

def evaluate_dwn(
    model,
    loader
):

    model.eval()


    y_true = []

    y_pred = []

    y_prob = []


    with torch.no_grad():

        for images, labels, paths in loader:

            images = images.to(
                device
            )


            outputs = model(
                images
            )


            probabilities = torch.softmax(
                outputs,
                dim=1
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            y_true.extend(
                labels.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_prob.extend(
                probabilities[:, 1]
                .cpu()
                .numpy()
            )


    y_true = np.array(
        y_true
    )

    y_pred = np.array(
        y_pred
    )

    y_prob = np.array(
        y_prob
    )


    acc = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    cm = confusion_matrix(
        y_true,
        y_pred
    )


    tn, fp, fn, tp = cm.ravel()


    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0
    )


    auc = roc_auc_score(
        y_true,
        y_prob
    )


    ap = average_precision_score(
        y_true,
        y_prob
    )


    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "specificity": specificity,
        "roc_auc": auc,
        "average_precision": ap,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob
    }


dwn_metrics = evaluate_dwn(
    dwn,
    test_loader
)


print(
    classification_report(
        dwn_metrics["y_true"],
        dwn_metrics["y_pred"],
        target_names=CLASS_NAMES,
        digits=4
    )
)

In [ ]:
# ============================================================
# TREINAMENTO YOLO26n-CLS
# ============================================================

yolo_model = YOLO(
    "yolo26n-cls.pt"
)


yolo_results = yolo_model.train(

    data=YOLO_DATASET,

    epochs=YOLO_EPOCHS,

    imgsz=IMG_SIZE,

    batch=BATCH_SIZE,

    device=0 if torch.cuda.is_available()
           else "cpu",

    workers=2,

    patience=5,

    project="sar_comparison",

    name="yolo26n",

    pretrained=True,

    verbose=True
)

In [ ]:
import os
import glob
from ultralytics import YOLO

# ============================================================
# PROCURAR AUTOMATICAMENTE O best.pt
# ============================================================

candidatos = glob.glob(
    "/content/**/best.pt",
    recursive=True
)

print("Arquivos best.pt encontrados:")

for p in candidatos:
    print(" -", p)

if len(candidatos) == 0:
    raise FileNotFoundError(
        "Nenhum best.pt foi encontrado. "
        "Verifique se o treinamento do YOLO terminou."
    )

# Se houver vários, pega o mais recente
best_yolo_path = max(
    candidatos,
    key=os.path.getmtime
)

print("\nMelhor modelo encontrado:")
print(best_yolo_path)

# ============================================================
# CARREGAR YOLO
# ============================================================

yolo_model = YOLO(best_yolo_path)

print("\nYOLO carregado com sucesso!")

In [ ]:
# ============================================================
# AVALIAÇÃO YOLO
# ============================================================

yolo_y_true = []

yolo_y_pred = []

yolo_y_prob = []


for path, label in zip(
    test_paths,
    test_labels
):

    result = yolo_model(
        path,
        imgsz=IMG_SIZE,
        verbose=False
    )[0]


    prediction = int(
        result.probs.top1
    )


    probabilities = (
        result.probs.data
        .cpu()
        .numpy()
    )


    if len(probabilities) >= 2:

        oil_probability = float(
            probabilities[1]
        )

    else:

        oil_probability = 0.0


    yolo_y_true.append(
        int(label)
    )

    yolo_y_pred.append(
        prediction
    )

    yolo_y_prob.append(
        oil_probability
    )


yolo_y_true = np.array(
    yolo_y_true
)

yolo_y_pred = np.array(
    yolo_y_pred
)

yolo_y_prob = np.array(
    yolo_y_prob
)


# ------------------------------------------------------------
# métricas
# ------------------------------------------------------------

yolo_accuracy = accuracy_score(
    yolo_y_true,
    yolo_y_pred
)

yolo_precision = precision_score(
    yolo_y_true,
    yolo_y_pred,
    zero_division=0
)

yolo_recall = recall_score(
    yolo_y_true,
    yolo_y_pred,
    zero_division=0
)

yolo_f1 = f1_score(
    yolo_y_true,
    yolo_y_pred,
    zero_division=0
)


yolo_cm = confusion_matrix(
    yolo_y_true,
    yolo_y_pred
)


tn, fp, fn, tp = yolo_cm.ravel()


yolo_specificity = (
    tn / (tn + fp)
    if tn + fp > 0
    else 0
)


yolo_roc_auc = roc_auc_score(
    yolo_y_true,
    yolo_y_prob
)


yolo_ap = average_precision_score(
    yolo_y_true,
    yolo_y_prob
)


print(
    classification_report(
        yolo_y_true,
        yolo_y_pred,
        target_names=CLASS_NAMES,
        digits=4
    )
)

In [ ]:
# ============================================================
# COMPARAÇÃO DWN vs YOLO
# ============================================================

comparison = pd.DataFrame({

    "DWN": [

        dwn_metrics["accuracy"],
        dwn_metrics["precision"],
        dwn_metrics["recall"],
        dwn_metrics["f1"],
        dwn_metrics["specificity"],
        dwn_metrics["roc_auc"],
        dwn_metrics["average_precision"]

    ],

    "YOLO26n": [

        yolo_accuracy,
        yolo_precision,
        yolo_recall,
        yolo_f1,
        yolo_specificity,
        yolo_roc_auc,
        yolo_ap

    ]

}, index=[

    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "ROC-AUC",
    "Average Precision"

])


display(
    (comparison * 100).round(2)
)

In [ ]:
# ============================================================
# MATRIZES DE CONFUSÃO
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)


# ------------------------------------------------------------
# DWN
# ------------------------------------------------------------

axes[0].imshow(
    dwn_metrics["cm"],
    cmap="Blues"
)

axes[0].set_title(
    "DWN"
)

axes[0].set_xlabel(
    "Predição"
)

axes[0].set_ylabel(
    "Real"
)

axes[0].set_xticks(
    [0, 1],
    CLASS_NAMES
)

axes[0].set_yticks(
    [0, 1],
    CLASS_NAMES
)


for i in range(2):

    for j in range(2):

        axes[0].text(
            j,
            i,
            dwn_metrics["cm"][i, j],
            ha="center",
            va="center",
            fontsize=18
        )


# ------------------------------------------------------------
# YOLO
# ------------------------------------------------------------

axes[1].imshow(
    yolo_cm,
    cmap="Greens"
)

axes[1].set_title(
    "YOLO26n"
)

axes[1].set_xlabel(
    "Predição"
)

axes[1].set_ylabel(
    "Real"
)

axes[1].set_xticks(
    [0, 1],
    CLASS_NAMES
)

axes[1].set_yticks(
    [0, 1],
    CLASS_NAMES
)


for i in range(2):

    for j in range(2):

        axes[1].text(
            j,
            i,
            yolo_cm[i, j],
            ha="center",
            va="center",
            fontsize=18
        )


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# GRÁFICO COMPARATIVO
# ============================================================

metric_names = [

    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Specificity",
    "ROC-AUC"

]


dwn_values = [

    dwn_metrics["accuracy"],
    dwn_metrics["precision"],
    dwn_metrics["recall"],
    dwn_metrics["f1"],
    dwn_metrics["specificity"],
    dwn_metrics["roc_auc"]

]


yolo_values = [

    yolo_accuracy,
    yolo_precision,
    yolo_recall,
    yolo_f1,
    yolo_specificity,
    yolo_roc_auc

]


x = np.arange(
    len(metric_names)
)


width = 0.35


plt.figure(
    figsize=(14, 6)
)


plt.bar(
    x - width/2,
    np.array(dwn_values) * 100,
    width,
    label="DWN"
)


plt.bar(
    x + width/2,
    np.array(yolo_values) * 100,
    width,
    label="YOLO26n"
)


plt.xticks(
    x,
    metric_names
)

plt.ylabel(
    "Score (%)"
)

plt.ylim(
    0,
    105
)

plt.title(
    "DWN vs YOLO26n — SAR Oil Spill"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# TAMANHO DOS MODELOS
# ============================================================

dwn_params = sum(
    p.numel()
    for p in dwn.parameters()
)


yolo_params = sum(
    p.numel()
    for p in yolo_model.model.parameters()
)


print(
    f"DWN parâmetros : {dwn_params:,}"
)

print(
    f"YOLO parâmetros: {yolo_params:,}"
)


print()

print(
    f"YOLO/DWN = "
    f"{yolo_params / dwn_params:.2f}x"
)

In [ ]:
# ============================================================
# TAMANHO DOS MODELOS
# ============================================================

# salvar DWN
torch.save(
    dwn.state_dict(),
    "dwn_sar.pt"
)


dwn_size = os.path.getsize(
    "dwn_sar.pt"
) / (1024 ** 2)


yolo_size = os.path.getsize(
    best_yolo_path
) / (1024 ** 2)


print(
    f"DWN : {dwn_size:.2f} MB"
)

print(
    f"YOLO: {yolo_size:.2f} MB"
)

In [ ]:
# ============================================================
# BENCHMARK DWN
# ============================================================

dwn.eval()


# usar algumas imagens
benchmark_images = test_dataset


n_benchmark = min(
    300,
    len(benchmark_images)
)


start = time.perf_counter()


with torch.no_grad():

    for i in range(
        n_benchmark
    ):

        image, label, path = (
            benchmark_images[i]
        )


        image = image.unsqueeze(
            0
        ).to(device)


        _ = dwn(
            image
        )


dwn_elapsed = (
    time.perf_counter()
    - start
)


dwn_ms = (
    dwn_elapsed /
    n_benchmark *
    1000
)


print(
    f"DWN: {dwn_ms:.3f} ms/imagem"
)

print(
    f"DWN: "
    f"{1000/dwn_ms:.2f} imagens/s"
)

In [ ]:
# ============================================================
# BENCHMARK YOLO
# ============================================================

n_benchmark = min(
    300,
    len(test_paths)
)


start = time.perf_counter()


for path in test_paths[
    :n_benchmark
]:

    _ = yolo_model(
        path,
        imgsz=IMG_SIZE,
        verbose=False
    )


yolo_elapsed = (
    time.perf_counter()
    - start
)


yolo_ms = (
    yolo_elapsed /
    n_benchmark *
    1000
)


print(
    f"YOLO26n: {yolo_ms:.3f} ms/imagem"
)

print(
    f"YOLO26n: "
    f"{1000/yolo_ms:.2f} imagens/s"
)

In [ ]:
# ============================================================
# DWN VS YOLO NA MESMA IMAGEM
# ============================================================

def predict_dwn_image(
    path
):

    image = Image.open(
        path
    ).convert(
        "L"
    )


    image = image.resize(
        (
            IMG_SIZE,
            IMG_SIZE
        )
    )


    array = np.array(
        image,
        dtype=np.float32
    ) / 255.0


    tensor = torch.tensor(
        array
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    ).to(device)


    dwn.eval()


    with torch.no_grad():

        output = dwn(
            tensor
        )

        probabilities = torch.softmax(
            output,
            dim=1
        )[0]


    prediction = int(
        torch.argmax(
            probabilities
        )
    )


    confidence = float(
        probabilities[prediction]
    )


    return (
        prediction,
        confidence
    )


def predict_yolo_image(
    path
):

    result = yolo_model(
        path,
        imgsz=IMG_SIZE,
        verbose=False
    )[0]


    prediction = int(
        result.probs.top1
    )


    confidence = float(
        result.probs.top1conf
    )


    return (
        prediction,
        confidence
    )


# ------------------------------------------------------------
# escolher imagens
# ------------------------------------------------------------

selected_indices = random.sample(
    range(len(test_paths)),
    min(
        12,
        len(test_paths)
    )
)


fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 12)
)


axes = axes.flatten()


for plot_i, idx in enumerate(
    selected_indices
):

    path = test_paths[idx]

    real = int(
        test_labels[idx]
    )


    dwn_pred, dwn_conf = (
        predict_dwn_image(
            path
        )
    )


    yolo_pred, yolo_conf = (
        predict_yolo_image(
            path
        )
    )


    image = Image.open(
        path
    )


    axes[plot_i].imshow(
        image,
        cmap="gray"
    )


    real_name = (
        "OIL"
        if real == 1
        else "NO OIL"
    )


    dwn_name = (
        "OIL"
        if dwn_pred == 1
        else "NO OIL"
    )


    yolo_name = (
        "OIL"
        if yolo_pred == 1
        else "NO OIL"
    )


    dwn_ok = (
        dwn_pred == real
    )


    yolo_ok = (
        yolo_pred == real
    )


    title = (

        f"REAL: {real_name}\n"

        f"DWN: {dwn_name} "
        f"({dwn_conf*100:.1f}%) "
        f"{'✓' if dwn_ok else '✗'}\n"

        f"YOLO: {yolo_name} "
        f"({yolo_conf*100:.1f}%) "
        f"{'✓' if yolo_ok else '✗'}"

    )


    axes[plot_i].set_title(
        title,
        fontsize=11
    )


    axes[plot_i].axis(
        "off"
    )


for i in range(
    len(selected_indices),
    len(axes)
):

    axes[i].axis(
        "off"
    )


plt.suptitle(
    "DWN vs YOLO26n — Mesmas imagens SAR",
    fontsize=18
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# TREINAMENTO E AVALIAÇÃO DA WISARD (AGORA COM LBP)
# ============================================================
print("\n--- INICIANDO WISARD (MAPEAMENTO 2D + LBP) ---")

# A MÁGICA ACONTECE AQUI: Trocamos a rede antiga pela nossa nova de 2048 parâmetros!
wisard_model = MultiScaleLBPWisard(classes=2, num_scales=4).to(device)

wisard_criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.7]).to(device))
wisard_optimizer = torch.optim.Adam(wisard_model.parameters(), lr=0.001)

wisard_model.train()
for epoch in range(WISARD_EPOCHS):
    running_loss = 0.0
    # CORREÇÃO AQUI: adicionamos o 'paths'
    for images, labels, paths in train_loader:
        images, labels = images.to(device), labels.to(device)
        wisard_optimizer.zero_grad()
        outputs = wisard_model(images)
        loss = wisard_criterion(outputs, labels)
        loss.backward()
        wisard_optimizer.step()
        running_loss += loss.item()
    print(f"WiSARD Época [{epoch+1}/{WISARD_EPOCHS}] Loss: {running_loss/len(train_loader):.4f}")

wisard_model.eval()
wisard_preds = []
with torch.no_grad():
    # CORREÇÃO AQUI: adicionamos o 'paths'
    for images, labels, paths in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = wisard_model(images)
        _, predicted = torch.max(outputs.data, 1)
        wisard_preds.extend(predicted.cpu().numpy())

# Calculando as métricas da WiSARD
wisard_acc = accuracy_score(test_labels, wisard_preds)
wisard_prec = precision_score(test_labels, wisard_preds, zero_division=0)
wisard_rec = recall_score(test_labels, wisard_preds, zero_division=0)
wisard_f1 = f1_score(test_labels, wisard_preds, zero_division=0)

tn_w, fp_w, fn_w, tp_w = confusion_matrix(test_labels, wisard_preds).ravel()
wisard_spec = tn_w / (tn_w + fp_w)
wisard_roc = roc_auc_score(test_labels, wisard_preds)

print(f"\nWiSARD Recall (Óleo): {wisard_rec*100:.2f}% | Acurácia: {wisard_acc*100:.2f}%")


# ============================================================
# GRÁFICO COMPARATIVO TRIPLO: DWN vs WiSARD vs YOLO26n
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

metric_names = ["Accuracy", "Precision", "Recall", "F1", "Specificity", "ROC-AUC"]

# Valores da DWN (antiga)
dwn_values = [dwn_metrics["accuracy"], dwn_metrics["precision"], dwn_metrics["recall"],
              dwn_metrics["f1"], dwn_metrics["specificity"], dwn_metrics["roc_auc"]]

# Valores da WiSARD (nova estrela)
wisard_values = [wisard_acc, wisard_prec, wisard_rec, wisard_f1, wisard_spec, wisard_roc]

# Valores do YOLO (peso pesado)
yolo_values = [yolo_accuracy, yolo_precision, yolo_recall, yolo_f1, yolo_specificity, yolo_roc_auc]

x = np.arange(len(metric_names))
width = 0.25 # Diminuímos a largura da barra para caberem três

plt.figure(figsize=(15, 6))

# Desenhando as 3 barras
plt.bar(x - width, np.array(dwn_values) * 100, width, label="DWN (Mapeamento 1D)", color='#1f77b4')
plt.bar(x, np.array(wisard_values) * 100, width, label="LBP-Conv-WiSARD (2048 parâmetros!)", color='#2ca02c')
plt.bar(x + width, np.array(yolo_values) * 100, width, label="YOLO26n (Pesado)", color='#ff7f0e')

plt.xticks(x, metric_names)
plt.ylabel("Score (%)")
plt.ylim(0, 105)
plt.title("Evolução das Arquiteturas — SAR Oil Spill (DWN vs WiSARD vs YOLO)")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DWN VS YOLO — SOMENTE IMAGENS COM ÓLEO
# ============================================================

def predict_dwn_image(path):

    image = Image.open(path).convert("L")

    image = image.resize(
        (
            IMG_SIZE,
            IMG_SIZE
        )
    )

    array = np.array(
        image,
        dtype=np.float32
    ) / 255.0

    tensor = torch.tensor(
        array
    ).unsqueeze(
        0
    ).unsqueeze(
        0
    ).to(device)

    dwn.eval()

    with torch.no_grad():

        output = dwn(tensor)

        probabilities = torch.softmax(
            output,
            dim=1
        )[0]

    prediction = int(
        torch.argmax(probabilities)
    )

    confidence = float(
        probabilities[prediction]
    )

    return prediction, confidence


def predict_yolo_image(path):

    result = yolo_model(
        path,
        imgsz=IMG_SIZE,
        verbose=False
    )[0]

    prediction = int(
        result.probs.top1
    )

    confidence = float(
        result.probs.top1conf
    )

    return prediction, confidence


# ============================================================
# SELECIONAR SOMENTE IMAGENS REAIS COM ÓLEO
# ============================================================

oil_indices = [
    i
    for i, label in enumerate(test_labels)
    if int(label) == 1
]


selected_indices = random.sample(
    oil_indices,
    min(
        12,
        len(oil_indices)
    )
)


# ============================================================
# CRIAR FIGURA
# ============================================================

fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 12)
)

axes = axes.flatten()


# ============================================================
# TESTE
# ============================================================

dwn_correct = 0
yolo_correct = 0


for plot_i, idx in enumerate(selected_indices):

    path = test_paths[idx]

    # Como filtramos somente óleo:
    real = 1


    # --------------------------------------------------------
    # DWN
    # --------------------------------------------------------

    dwn_pred, dwn_conf = predict_dwn_image(
        path
    )


    # --------------------------------------------------------
    # YOLO
    # --------------------------------------------------------

    yolo_pred, yolo_conf = predict_yolo_image(
        path
    )


    # --------------------------------------------------------
    # IMAGEM
    # --------------------------------------------------------

    image = Image.open(path)


    axes[plot_i].imshow(
        image,
        cmap="gray"
    )


    # --------------------------------------------------------
    # NOMES
    # --------------------------------------------------------

    dwn_name = (
        "OIL"
        if dwn_pred == 1
        else "NO OIL"
    )


    yolo_name = (
        "OIL"
        if yolo_pred == 1
        else "NO OIL"
    )


    # --------------------------------------------------------
    # ACERTOS
    # --------------------------------------------------------

    dwn_ok = (
        dwn_pred == real
    )

    yolo_ok = (
        yolo_pred == real
    )


    if dwn_ok:
        dwn_correct += 1

    if yolo_ok:
        yolo_correct += 1


    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    title = (

        "REAL: OIL\n"

        f"DWN: {dwn_name} "
        f"({dwn_conf * 100:.1f}%) "
        f"{'✓' if dwn_ok else '✗'}\n"

        f"YOLO: {yolo_name} "
        f"({yolo_conf * 100:.1f}%) "
        f"{'✓' if yolo_ok else '✗'}"

    )


    axes[plot_i].set_title(
        title,
        fontsize=11
    )

    axes[plot_i].axis("off")


# ============================================================
# OCULTAR QUADROS SOBRANDO
# ============================================================

for i in range(
    len(selected_indices),
    len(axes)
):

    axes[i].axis("off")


# ============================================================
# RESULTADOS
# ============================================================

total = len(selected_indices)


if total > 0:

    dwn_acc = (
        dwn_correct / total
    ) * 100

    yolo_acc = (
        yolo_correct / total
    ) * 100


    print(
        f"Total de imagens com óleo: {total}"
    )

    print(
        f"DWN acertou: "
        f"{dwn_correct}/{total} "
        f"({dwn_acc:.1f}%)"
    )

    print(
        f"YOLO acertou: "
        f"{yolo_correct}/{total} "
        f"({yolo_acc:.1f}%)"
    )


# ============================================================
# MOSTRAR
# ============================================================

plt.suptitle(
    "DWN vs YOLO — Somente imagens SAR COM ÓLEO",
    fontsize=18
)

plt.tight_layout()

plt.show()

In [ ]:
%%bash
find /content/oil_yolov4 -type f \
  \( -name "*.py" -o -name "*.ipynb" \) \
  2>/dev/null | grep -Ei \
  'dwnn|weightless|differentiable|neural|winn|wlnn' | head -100

In [ ]:
# ============================================================
# LABORATÓRIO ISOLADO: KALWENN & BLOOMWISARD (Artigo SBBD 2026)
# ============================================================
import numpy as np
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score
import wisardpkg as wp
import torch.nn.functional as F # <-- Importação adicionada para proteger a RAM

print("\n--- INICIANDO LABORATÓRIO DE REDES PURAS (KalWeNN / BloomWisard) ---")

# 1. CLASSE KALWENN (Implementação do Artigo)
class KalWeNN:
    def __init__(self, address_size, q=1e-3, r=1e-1, p_init=1.0):
        self.address_size = address_size
        self.q = q
        self.r = r
        self.p_init = p_init
        self.memory = {}

    def _get_tuples(self, X_bin_flat):
        return [tuple(X_bin_flat[i:i+self.address_size])
                for i in range(0, len(X_bin_flat) - len(X_bin_flat)%self.address_size, self.address_size)]

    def train(self, X, y):
        for i, x_bin in enumerate(X):
            label = y[i]
            if label not in self.memory:
                self.memory[label] = []
            tuples = self._get_tuples(x_bin)
            if len(self.memory[label]) == 0:
                for _ in range(len(tuples)):
                    self.memory[label].append({})
            for ram_idx, t in enumerate(tuples):
                ram = self.memory[label][ram_idx]
                if t not in ram:
                    ram[t] = {'x': 0.0, 'P': self.p_init}
                x_prev = ram[t]['x']
                p_prev = ram[t]['P']
                p_pred = p_prev + self.q
                k = p_pred / (p_pred + self.r)
                x_new = x_prev + k * (1.0 - x_prev)
                p_new = (1.0 - k) * p_pred
                ram[t]['x'] = x_new
                ram[t]['P'] = p_new

    def classify(self, X):
        predictions = []
        for x_bin in X:
            tuples = self._get_tuples(x_bin)
            scores = {label: 0.0 for label in self.memory.keys()}
            for label, rams in self.memory.items():
                for ram_idx, t in enumerate(tuples):
                    if ram_idx < len(rams) and t in rams[ram_idx]:
                        scores[label] += rams[ram_idx][t]['x']
            best_class = max(scores, key=scores.get)
            predictions.append(best_class)
        return predictions

# 2. FUNÇÃO DE TERMÔMETRO
def aplicar_termometro(X, tamanho_termometro, tipo='distributive'):
    limiares = np.linspace(np.min(X), np.max(X), tamanho_termometro)
    X_bin = []
    for imagem in X:
        img_bin = []
        for valor in imagem:
            bits = [0] * tamanho_termometro
            for i, limiar in enumerate(limiares):
                if valor >= limiar:
                    bits[i] = 1
                else:
                    break
            if tipo == 'distributive':
                bits = bits[0::2] + bits[1::2]
            img_bin.extend(bits)
        X_bin.append(img_bin)
    return X_bin

# 3. EXTRAÇÃO DOS DADOS DO PYTORCH (COM PROTEÇÃO DE RAM!)
print("Convertendo tensores do PyTorch e reduzindo resolução para poupar RAM...")
def extrair_dados_do_loader(loader, image_size=32):
    X, Y = [], []
    for images, labels, _ in loader:
        # Encolhe a imagem de 128x128 para 32x32 para não estourar a memória
        images_resized = F.adaptive_avg_pool2d(images, (image_size, image_size))
        X.append(images_resized.cpu().numpy().reshape(images_resized.shape[0], -1))
        Y.extend(labels.cpu().numpy().astype(str))
    return np.vstack(X), Y

X_train_np, y_train_np = extrair_dados_do_loader(train_loader)
X_test_np, y_test_np = extrair_dados_do_loader(test_loader)

# 4. GRID SEARCH COMPACTO (KalWeNN vs WiSARD Clássica)
parametros_wisard = {
    'address_size': [11, 20],
    'thermometer_size': [16],
    'thermometer_type': ['simple', 'distributive'],
    'modelo': ['KalWeNN', 'Wisard_Pura']
}

grid = ParameterGrid(parametros_wisard)
print(f"Total de combinações para testar: {len(grid)}")

melhor_f1_puro = 0
melhor_config_pura = None

for config in grid:
    print(f"\n[+] Treinando: {config['modelo']} | Tupla: {config['address_size']} | Termômetro: {config['thermometer_type']}")

    # Binarização
    X_train_bin = aplicar_termometro(X_train_np, config['thermometer_size'], config['thermometer_type'])
    X_test_bin  = aplicar_termometro(X_test_np, config['thermometer_size'], config['thermometer_type'])

    # Instanciação Corrigida
    if config['modelo'] == 'Wisard_Pura':
        # Puxa a WiSARD padrão da biblioteca oficial (extremamente rápida)
        modelo = wp.Wisard(config['address_size'], ignoreZero=False)
    else:
        # Puxa a nossa KalWeNN
        modelo = KalWeNN(config['address_size'], q=1e-3, r=1e-1)

    # Treino e Teste
    modelo.train(X_train_bin, y_train_np)
    predicoes = modelo.classify(X_test_bin)

    y_test_int = [int(y) for y in y_test_np]
    predicoes_int = [int(p) for p in predicoes]
    f1 = f1_score(y_test_int, predicoes_int, pos_label=1)

    print(f"    F1-Score atingido: {f1:.4f}")
    if f1 > melhor_f1_puro:
        melhor_f1_puro = f1
        melhor_config_pura = config

print("\n" + "="*50)
print(f"🏆 VENCEDOR DO LABORATÓRIO: {melhor_config_pura['modelo']}")
print(f"🥇 F1-SCORE: {melhor_f1_puro:.4f}")
print("="*50)
print("Siga agora para o bloco original (LBP-Conv-WiSARD) logo abaixo!\n")

In [ ]:
from codecarbon import EmissionsTracker
import torch

print("Iniciando o Benchmark de Energia (Inferência)...")

# =======================================================
# 1. MEDINDO A ENERGIA DA DWN (1D Clássica)
# =======================================================
#print("\n[1/3] Medindo DWN 1D...")
#tracker_dwn = EmissionsTracker(project_name="DWN_Inference", log_level="error")
#tracker_dwn.start()

#dwn_model.eval()
#with torch.no_grad():
    #for images, labels, paths in test_loader:
        #images = images.to(device)
        #_ = dwn_model(images)

#tracker_dwn.stop()
#energia_dwn_kwh = tracker_dwn.final_emissions_data.energy_consumed

# =======================================================
# 2. MEDINDO A ENERGIA DA WiSARD (Multi-Scale LBP)
# =======================================================
print("[2/3] Medindo WiSARD LBP Multi-Scale...")
tracker_wisard = EmissionsTracker(project_name="WiSARD_Inference", log_level="error")
tracker_wisard.start()

wisard_model.eval()
with torch.no_grad():
    for images, labels, paths in test_loader:
        images = images.to(device)
        _ = wisard_model(images)

tracker_wisard.stop()
energia_wisard_kwh = tracker_wisard.final_emissions_data.energy_consumed

# =======================================================
# 3. MEDINDO A ENERGIA DO YOLO26n
# =======================================================
# ATENÇÃO: Ajuste a linha "_ = yolo_model(images)" caso o seu
# YOLO use um formato diferente de inferência (como model.predict)
tracker_yolo = EmissionsTracker(project_name="YOLO_Inference", log_level="error")
tracker_yolo.start()

# Executa a inferência do YOLO sem calcular gradientes
with torch.no_grad():
    for images, labels, paths in test_loader:
        # Passa as rotas e desliga o texto no terminal para ir rápido
        _ = yolo_model(paths, verbose=False)

tracker_yolo.stop()
energia_yolo_kwh = tracker_yolo.final_emissions_data.energy_consumed
# =======================================================
# PLACAR FINAL DE ENERGIA
# =======================================================
print(f"\n=============================================")
print(f"⚡ CONSUMO ENERGÉTICO TOTAL (INFERÊNCIA) ⚡")
print(f"=============================================")
print(f"DWN 1D (Mais leve):    {energia_dwn_kwh:.8f} kWh")
print(f"WiSARD 2D (Ideal):     {energia_wisard_kwh:.8f} kWh")
print(f"YOLO26n (Pesado):      {energia_yolo_kwh:.8f} kWh")
print(f"=============================================")

In [ ]:
import os
import pickle
import torch
from collections import Counter

print("="*50)
print("📊 RAIO-X DO DATASET")
print("="*50)

# 1. CONTAGEM DO DATASET
contador_classes = Counter()
total_imagens = 0

# Varre o DataLoader de treino
for _, labels, _ in train_loader:
    contador_classes.update(labels.cpu().numpy().tolist())
    total_imagens += len(labels)

# Varre o DataLoader de teste
for _, labels, _ in test_loader:
    contador_classes.update(labels.cpu().numpy().tolist())
    total_imagens += len(labels)

print(f"Total absoluto de imagens processadas: {total_imagens}")
# Assumindo que 1 seja Óleo e 0 seja Água/Sem Óleo (ajuste se for o inverso)
print(f"-> Imagens COM ÓLEO (Classe 1): {contador_classes.get(1, 0)}")
print(f"-> Imagens SEM ÓLEO (Classe 0): {contador_classes.get(0, 0)}")


print("\n" + "="*50)
print("💾 PEGADA DE MEMÓRIA DOS MODELOS (SWaP - Size)")
print("="*50)

def medir_tamanho_arquivo(caminho, nome_modelo):
    tamanho_bytes = os.path.getsize(caminho)
    tamanho_mb = tamanho_bytes / (1024 * 1024)
    tamanho_kb = tamanho_bytes / 1024

    if tamanho_mb > 1.0:
        print(f"[{nome_modelo}] ocupará: {tamanho_mb:.2f} MB na RAM/Disco do Satélite")
    else:
        print(f"[{nome_modelo}] ocupará: {tamanho_kb:.2f} KB na RAM/Disco do Satélite")

# 2. MEDINDO A LBP-Conv-WiSARD (Seu Modelo PyTorch)
# Salvamos os pesos num arquivo temporário para medir
torch.save(wisard_model.state_dict(), 'temp_lbp_wisard.pt')
medir_tamanho_arquivo('temp_lbp_wisard.pt', 'LBP-Conv-WiSARD')

# 3. MEDINDO A KalWeNN / BloomWisard (Rede Pura)
try:
    with open('temp_kalwenn.pkl', 'wb') as f:
        pickle.dump(modelo, f)
    medir_tamanho_arquivo('temp_kalwenn.pkl', 'KalWeNN / WiSARD Pura')
except TypeError:
    # Se o modelo for da wisardpkg (C++), medimos a string JSON dele
    tamanho_bytes = len(modelo.json())
    tamanho_kb = tamanho_bytes / 1024
    print(f"[WiSARD Pura (C++)] ocupará: {tamanho_kb:.2f} KB na RAM/Disco do Satélite")

# 4. MEDINDO O YOLO (Se ele estiver instanciado)
# Se você tiver carregado o modelo YOLO (ex: yolo_model = YOLO('yolov8n.pt')),
# podemos medir o arquivo oficial dele.
try:
    medir_tamanho_arquivo('yolov8n.pt', 'YOLOv8n (Referência)')
except FileNotFoundError:
    print("[YOLOv8n] Arquivo .pt não encontrado no diretório atual para medir.")

# Limpeza dos arquivos temporários
os.remove('temp_lbp_wisard.pt')
os.remove('temp_kalwenn.pkl')

In [ ]:
from ultralytics import YOLO

print("="*50)
print("📥 MEDINDO (YOLO)")
print("="*50)

# Força o download/criação do arquivo yolov8n.pt na pasta atual
modelo_yolo = YOLO('yolov8n.pt')

# Mede o arquivo que acabou de ser baixado
medir_tamanho_arquivo('yolov8n.pt', 'YOLOv8n (Referência)')

In [ ]:
import time
from sklearn.metrics import accuracy_score, recall_score, f1_score
import torch

print("="*50)
print("🎯 MÉTRICAS FINAIS: LBP-Conv-WiSARD")
print("="*50)

# Extração de Predições
y_true = []
y_pred = []
tempos_inferencia = []

wisard_model.eval() # Coloca seu modelo em modo de avaliação (ajuste o nome se precisar)
with torch.no_grad():
    for images, labels, _ in test_loader:
        # Cronometra o tempo que a rede leva para classificar
        inicio = time.time()

        saidas = wisard_model(images) # Passe a imagem pela sua rede
        _, predicoes = torch.max(saidas, 1)

        fim = time.time()

        # Calcula o tempo por imagem no lote (em milissegundos)
        tempo_por_imagem_ms = ((fim - inicio) / len(labels)) * 1000
        tempos_inferencia.append(tempo_por_imagem_ms)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicoes.cpu().numpy())

# Cálculos do sklearn
acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred, pos_label=1)
f1  = f1_score(y_true, y_pred, pos_label=1)
tempo_medio_ms = sum(tempos_inferencia) / len(tempos_inferencia)

print(f"-> Acurácia: {acc*100:.2f}%")
print(f"-> Recall:   {rec*100:.2f}%")
print(f"-> F1-Score: {f1*100:.2f}%")
print(f"-> Tempo de Inferência (por imagem): {tempo_medio_ms:.4f} ms")